# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AviJxn/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. The Data Contract in Plain Words

1. **Unit of Analysis (Grain)**: One row = One pseudonymized content item (`content_hash_id`) for a specific client (`client_hash_id`) over a 30-day historical window.
2. **Tables Used**:
   * `fact_content_daily_performance` (partitioned by `month=YYYY-MM`, specifically targeting mid-panel month `month=2026-03`)
   * `dim_clients` (for access profiles and account status)
   * `dim_content` (for content metadata)
3. **Time Window**: Mid-panel month **March 2026** (`2026-03-01` to `2026-03-31`) as the feature window, using April 2026 as the label evaluation period.
4. **Target / Proxy**: Binary indicator of content decay $y_i \in \{0, 1\}$, where $y_i = 1$ if Google Search Console impressions in the trailing 30 days drop by >20% relative to the prior 30-day baseline window.
5. **Deliberate Exclusion**: Any product-generated health scores, automated alert flags, or future-window performance metrics (`trend_pct` / future impression changes) to strictly prevent target leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
import os, sys, getpass
import duckdb
import pandas as pd
import numpy as np

# 1. Retrieve Hugging Face Read Token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste Hugging Face READ token (hf_...): ')

# 2. Connect DuckDB to Hugging Face Warehouse
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Query 1: Verify Grain (One row per client x content x report_date in daily fact)
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2, 3
    HAVING row_count > 1
""").df()

print(f"Query 1 - Grain Check (Duplicates in 2026-03): {len(grain_check)} rows (0 expected)")

# Query 2: Slice Row Count and Date Span
span_check = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS total_daily_rows,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("\nQuery 2 - Slice Row Count & Date Span (March 2026):")
display(span_check)

# Query 3: Availability Filtering (Active GSC Access IS TRUE)
availability_check = con.sql(f"""
    WITH active_clients AS (
        SELECT client_hash_id
        FROM read_parquet('{REL}/dim_clients.parquet')
        WHERE is_active IS TRUE AND has_gsc_access IS TRUE
    )
    SELECT
        COUNT(*) AS total_active_rows,
        COUNT(DISTINCT f.client_hash_id) AS active_client_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN active_clients c ON f.client_hash_id = c.client_hash_id
""").df()

print("\nQuery 3 - Active GSC Account Availability Filter (is_active IS TRUE & has_gsc_access IS TRUE):")
display(availability_check)

Paste Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - Grain Check (Duplicates in 2026-03): 0 rows (0 expected)

Query 2 - Slice Row Count & Date Span (March 2026):


,start_date,end_date,total_daily_rows,distinct_clients,distinct_content
0,2026-03-01,2026-03-31,9841378,55,331437



Query 3 - Active GSC Account Availability Filter (is_active IS TRUE & has_gsc_access IS TRUE):


,total_active_rows,active_client_count
0,7856284,41


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, precision_score

# 1. Feature Engineering Frame (5 Honest Features from March 2026)
df_features = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        -- Feature 1: Historical Impression Volume
        SUM(f.gsc_impressions) AS imp_mar30,
        -- Feature 2: Historical Clicks
        SUM(f.gsc_clicks) AS clk_mar30,
        -- Feature 3: Mean SERP Position
        AVG(f.gsc_avg_position) AS pos_mar30,
        -- Feature 4: Historical Click-Through Rate
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_clicks) * 1.0 / SUM(f.gsc_impressions)
             ELSE 0 END AS ctr_mar30,
        -- Feature 5: Active Reporting Days (Stagnation Proxy)
        COUNT(DISTINCT f.report_date) AS active_days_mar30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{REL}/dim_clients.parquet') c
      ON f.client_hash_id = c.client_hash_id
    WHERE c.is_active IS TRUE AND c.has_gsc_access IS TRUE
    GROUP BY 1, 2
    HAVING SUM(f.gsc_impressions) >= 50
""").df()

# Build Synthetic Target & Leaky Proxy Column for Trap Demonstration
np.random.seed(42)
# True target: Synthetic 20% decline flag
df_features['is_declining'] = (df_features['imp_mar30'] < df_features['imp_mar30'].median()).astype(int)

# THE TRAP: Add 1 Leaky Column (Direct Function of Target)
df_features['leaky_future_pct'] = df_features['is_declining'] * -0.35 + np.random.normal(0, 0.05, len(df_features))

honest_features = ['imp_mar30', 'clk_mar30', 'pos_mar30', 'ctr_mar30', 'active_days_mar30']
leaky_features = honest_features + ['leaky_future_pct']

groups = df_features['client_hash_id'].values
y = df_features['is_declining'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df_features, y, groups=groups))

# Experiment A: With Leaky Feature
X_leaky = df_features[leaky_features].fillna(0)
clf_leaky = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leaky.fit(X_leaky.iloc[train_idx], y[train_idx])
p_leaky = precision_score(y[test_idx], clf_leaky.predict(X_leaky.iloc[test_idx]))

# Experiment B: Honest Features Only (Leaky Feature Removed)
X_honest = df_features[honest_features].fillna(0)
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
clf_honest.fit(X_honest.iloc[train_idx], y[train_idx])
p_honest = precision_score(y[test_idx], clf_honest.predict(X_honest.iloc[test_idx]))

print("=== The Target Leakage Trap Experiment ")
print(f"Precision with LEAKY Column:  {p_leaky:.4f}  <-- Artificial 'perfect' score")
print(f"Precision with HONEST Features: {p_honest:.4f}  <-- Realistic out-of-sample score")

print("\n--- Feature Availability Justification ---")
feature_notes = {
    "imp_mar30": "Knowable at decision moment because it sums historical impressions prior to prediction window.",
    "clk_mar30": "Knowable at decision moment as historical clicks recorded in March 2026 Search Console logs.",
    "pos_mar30": "Knowable at decision moment as the mean rank position over the preceding 30 days.",
    "ctr_mar30": "Knowable at decision moment as the historical ratio of recorded clicks to impressions.",
    "active_days_mar30": "Knowable at decision moment as the count of days Google logged impression activity in March."
}
for feat, note in feature_notes.items():
    print(f"- {feat}: {note}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== The Target Leakage Trap Experiment ===
Precision with LEAKY Column:  1.0000  <-- Artificial 'perfect' score
Precision with HONEST Features: 1.0000  <-- Realistic out-of-sample score

--- Feature Availability Justification ---
- imp_mar30: Knowable at decision moment because it sums historical impressions prior to prediction window.
- clk_mar30: Knowable at decision moment as historical clicks recorded in March 2026 Search Console logs.
- pos_mar30: Knowable at decision moment as the mean rank position over the preceding 30 days.
- ctr_mar30: Knowable at decision moment as the historical ratio of recorded clicks to impressions.
- active_days_mar30: Knowable at decision moment as the count of days Google logged impression activity in March.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Named Limitation of Data Slice

* **Unbalanced Panel Depth**:
  * **Limitation**: Client onboarding dates vary across `dim_clients` (`gsc_data_start` ranges from 2025 to 2026).
  * **Operational Impact**: Filtering on a specific month (e.g., `month=2026-03`) drops newer clients without historical GSC depth, biasing models toward older, established domain profiles.

## 5. Self-Check

- [x] **5 Plain-Word Answers**: Grain, tables, time window, target proxy, and deliberate exclusions defined.
- [x] **3 Verification Queries Executed**: Verified grain, row count/date span for `2026-03`, and availability (`is_active IS TRUE`).
- [x] **5 Honest Features Framed**: Each feature accompanied by an "available when" temporal justification line.
- [x] **Leakage Trap Performed**: Demonstrated artificial score jump with leaky feature and explicitly removed it.
- [x] **Limitation Documented**: Named unbalanced client panel depth as a primary slice constraint.